<a href="https://colab.research.google.com/github/kulkarnishravan/notebook_lab/blob/main/hyde.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Install all necessary LangChain components and FAISS
!pip install -q langchain langchain-text-splitters langchain-openai langchain-community faiss-cpu tiktoken

import os
from google.colab import userdata
from getpass import getpass

# Safely set your OpenAI API key
try:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except Exception as e:
    print("Could not retrieve key from Colab Secrets automatically.")
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key: ")

In [7]:
# Install required libraries
!pip install -q langchain langchain-openai faiss-cpu tiktoken

import os
from google.colab import userdata
from getpass import getpass

# Safely set your OpenAI API key
try:
    # Try to retrieve the key from Colab Secrets
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except Exception as e:
    print("Could not retrieve key from Colab Secrets automatically.")
    # Fallback to interactive input if the secret is missing or times out
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key: ")

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Sample technical text that lacks direct keyword repetition for abstract queries
sample_text = """
Project Orion Architecture Overview:
The system core relies on an asynchronous event-driven message mesh rather than a traditional synchronous REST API layout.
Microservices communicate via an Apache Kafka broker instance configured with explicit topic partitions to guarantee ordering semantics.
State management is isolated entirely from compute nodes; persistence is delegated to a distributed multi-region CockroachDB cluster using strong consistency configurations.
For disaster recovery, continuous Write-Ahead Logs (WAL) are mirrored to an immutable cloud object storage bucket every 60 seconds.
Caching layers utilize Redis Enterprise clusters with active-active geo-replication enabled to suppress cross-region latency spikes down to sub-15ms thresholds.
"""

# 1. Chunking the text
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30
)
docs = text_splitter.create_documents([sample_text])

# 2. Setup Embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 3. Build FAISS Vector Database
vectorstore = FAISS.from_documents(docs, embeddings)
print(f"Successfully created {len(docs)} text chunks and loaded them into FAISS.")

Successfully created 8 text chunks and loaded them into FAISS.


In [9]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Updated HyDE prompt that explicitly targets database structure and consistency
hyde_prompt = PromptTemplate(
    input_variables=["question"],
    template="""Write a technical architecture paragraph explaining how database storage, multi-region replication, and consistency configurations handle the following query. Do not focus on caching.

    Query: {question}

    Paragraph:"""
)

hyde_chain = hyde_prompt | llm | StrOutputParser()

In [10]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

difficult_query = "How does this platform handle cross-region database consistency and latency reduction?"

# --- GET K DYNAMICALLY FROM USER ---
try:
    user_k = int(input("Enter the top K value for document retrieval (e.g., 1, 2, 3): "))
except ValueError:
    user_k = 1  # Fallback default if input is invalid
    print("Invalid input received. Defaulting K to 1.")

print(f"Configured retriever with k={user_k}")

print("=" * 80)
print(f"EXECUTING RETRIEVAL COMPARISON (k={user_k})")
print("=" * 80)

# --- METHOD 1: Standard RAG (Raw Query) ---
standard_retriever = vectorstore.as_retriever(search_kwargs={"k": user_k})
standard_docs = standard_retriever.invoke(difficult_query)
standard_context = "\n".join([d.page_content for d in standard_docs])

print(f"\n[Standard RAG Retrieved Chunks (k={user_k})]:")
for i, d in enumerate(standard_docs):
    print(f"  {i+1}. {d.page_content.strip()}")

# --- METHOD 2: HyDE RAG (Hypothetical Document) ---
print("\n-> Generating HyDE hypothetical document...")
hypothetical_document = hyde_chain.invoke({"question": difficult_query})

hyde_retriever = vectorstore.as_retriever(search_kwargs={"k": user_k})
hyde_docs = hyde_retriever.invoke(hypothetical_document)
hyde_context = "\n".join([d.page_content for d in hyde_docs])

print(f"\n[HyDE RAG Retrieved Chunks (k={user_k})]:")
for i, d in enumerate(hyde_docs):
    print(f"  {i+1}. {d.page_content.strip()}")

# --- GENERATE FINAL ANSWERS ---
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the user query strictly using the provided context:\n\n{context}"),
    ("human", "{query}")
])
qa_chain = qa_prompt | llm | StrOutputParser()

standard_answer = qa_chain.invoke({"context": standard_context, "query": difficult_query})
hyde_answer = qa_chain.invoke({"context": hyde_context, "query": difficult_query})

print("\n" + "=" * 80)
print("FINAL COMPARISON")
print("=" * 80)
print(f">> STANDARD RAG ANSWER:\n{standard_answer}\n")
print("-" * 80)
print(f">> HyDE RAG ANSWER:\n{hyde_answer}")
print("=" * 80)

Enter the top K value for document retrieval (e.g., 1, 2, 3): 1
Configured retriever with k=1
EXECUTING RETRIEVAL COMPARISON (k=1)

[Standard RAG Retrieved Chunks (k=1)]:
  1. Caching layers utilize Redis Enterprise clusters with active-active geo-replication enabled to suppress cross-region latency spikes down to sub-15ms

-> Generating HyDE hypothetical document...

[HyDE RAG Retrieved Chunks (k=1)]:
  1. State management is isolated entirely from compute nodes; persistence is delegated to a distributed multi-region CockroachDB cluster using strong

FINAL COMPARISON
>> STANDARD RAG ANSWER:
The platform handles cross-region database consistency and latency reduction by utilizing Redis Enterprise clusters with active-active geo-replication enabled. This setup helps to suppress cross-region latency spikes down to sub-15ms.

--------------------------------------------------------------------------------
>> HyDE RAG ANSWER:
The platform handles cross-region database consistency through t